# R で学ぶ時系列分析 入門チュートリアル

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** の R カーネル（xeus-r）で、
**追加パッケージなし（base R の `stats` と `datasets` だけ）** で時系列分析の基本を学ぶチュートリアルです。

R には時系列のためのクラス `ts` と、分解・自己相関・ARIMA・指数平滑といった
基本的な分析道具が最初から入っています。月次売上や年次データの分析なら、これだけで十分始められます。

## 対象者
- R の基本文法（`r/r_beginner_tutorial.ipynb`）を終えた方
- 売上・アクセス数・経済指標など「時間とともに変わるデータ」を分析したい方

## このチュートリアルで学ぶこと
1. ts オブジェクト — 時系列専用のデータ形式
2. 時系列の可視化（plot / ts.plot）
3. ラグと差分（diff / lag）— 変化量と成長率
4. 移動平均（stats::filter）— なめらかな傾向を見る
5. 分解（decompose / stl）— トレンド・季節・残差に分ける
6. 自己相関（acf / pacf）— 過去との相関を測る
7. ARIMA モデル（arima / predict）— モデルを当てはめて予測する
8. 指数平滑法（HoltWinters）— 実務で定番の予測法
9. ランダムウォーク — 「見せかけのトレンド」に注意
10. 総合演習 — 月次売上の分解と 12 か月先予測

例には R 内蔵の **AirPassengers**（1949–1960 年の月次国際航空旅客数）と
**co2**（月次大気 CO2 濃度）を使います。

> **注意**：この環境ではグラフ内の日本語が文字化けするため、グラフのタイトルや軸ラベルは英語で書きます。

In [ ]:
# タイムゾーンとグラフの設定（JupyterLite 向けのおまじない）
Sys.setenv(TZ = "Asia/Tokyo")
options(repr.plot.width = 7, repr.plot.height = 4.5, repr.plot.res = 100, jupyter.plot_scale = 1)

cat("R のバージョン:", R.version.string, "\n")
cat("この章で使う内蔵データ: AirPassengers, co2, Nile\n")

---
## 1. ts オブジェクト — 時系列専用のデータ形式

`ts()` は数値ベクトルに「**いつから・どの頻度で** 観測したか」という時間情報を付けたオブジェクトです。

```r
ts(データ, start = c(開始年, 開始期), frequency = 1年あたりの観測数)
```

- `frequency = 12`：月次、`4`：四半期、`1`：年次
- `start = c(2024, 1)`：2024 年の第 1 期（月次なら 1 月）から

主な補助関数：`start()` / `end()`（期間）、`frequency()`（頻度）、
`window()`（期間の切り出し）、`time()`（各観測の時刻）、`cycle()`（何期目か）。

In [ ]:
# 月次売上（24 か月分）を ts にする
set.seed(1)
x <- ts(round(100 + 1:24 * 2 + rnorm(24, 0, 5)), start = c(2024, 1), frequency = 12)
print(x)          # 月次の ts は「年 × 月」の表形式で表示される

cat("開始:", start(x), " 終了:", end(x), " 頻度:", frequency(x), "\n")

In [ ]:
# 内蔵データ AirPassengers（月次・1949〜1960 年）
print(AirPassengers)

# window()：期間を切り出す（1958 年以降だけ）
window(AirPassengers, start = c(1958, 1))

### 練習問題 1

四半期売上のベクトル `c(120, 135, 160, 210, 128, 149, 175, 231)` を、
**2024 年第 1 四半期開始・四半期データ** の ts オブジェクト `q` にして表示し、
さらに `window()` で **2025 年の 4 つの値だけ** を切り出してください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```r
q <- ts(c(120, 135, 160, 210, 128, 149, 175, 231),
        start = c(2024, 1), frequency = 4)
print(q)
window(q, start = c(2025, 1))
```

</details>

---
## 2. 時系列の可視化

ts オブジェクトを `plot()` に渡すだけで、横軸が時間の折れ線グラフになります。
複数の系列を重ねるときは `ts.plot()` が便利です。

AirPassengers を見ると、**右上がりのトレンド**、**毎年夏に高くなる季節性**、
そして **振れ幅が年々大きくなる**（分散が増える）ことが読み取れます。
振れ幅が水準に比例して大きくなる系列は、**対数をとる** と扱いやすくなります。

In [ ]:
plot(AirPassengers, main = "International air passengers (1949-1960)",
     ylab = "Passengers (1000s)", col = "steelblue", lwd = 2)

In [ ]:
# 原系列と対数系列を並べて比較（対数にすると振れ幅がほぼ一定になる）
par(mfrow = c(1, 2))
plot(AirPassengers, main = "Original", ylab = "Passengers", col = "steelblue")
plot(log(AirPassengers), main = "Log scale", ylab = "log(Passengers)", col = "tomato")
par(mfrow = c(1, 1))

### 練習問題 2

内蔵データ `co2`（月次 CO2 濃度）について、

1. 全期間をプロットしてください（タイトル "Atmospheric CO2"、y 軸 "ppm"）
2. `window()` で 1990 年以降だけを切り出してプロットしてください

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```r
plot(co2, main = "Atmospheric CO2", ylab = "ppm", col = "steelblue")
plot(window(co2, start = c(1990, 1)),
     main = "Atmospheric CO2 (1990-)", ylab = "ppm", col = "tomato")
```

</details>

---
## 3. ラグと差分 — 変化量と成長率

- `diff(x)`：隣り合う値の差（前期からの **変化量**）。`diff(x, lag = 12)` なら **前年同月差**
- `diff(log(x))`：対数の差 ≒ **成長率**（×100 で %）
- `lag(x, -1)`：系列を 1 期ずらす（`ts.intersect()` と組み合わせると「今期と前期」を並べられる）

トレンドのある系列も、差分をとると「変化」の系列になり、平均のまわりを行き来する形
（**定常** に近い形）になります。これは後の ARIMA の「I（差分）」につながる操作です。

In [ ]:
# 前月差と前年同月差
print(head(diff(AirPassengers), 12))              # 前月からの変化量
print(head(diff(AirPassengers, lag = 12), 12))    # 前年同月からの変化量

# 今期と前期を並べて見る（lag(x, -1) が「1 期前の値」）
head(ts.intersect(now = AirPassengers, prev = lag(AirPassengers, -1)))

In [ ]:
# 成長率（%）：対数差分 × 100
growth <- diff(log(AirPassengers)) * 100
plot(growth, main = "Monthly growth rate (%)", ylab = "%", col = "steelblue")
abline(h = 0, col = "gray40", lty = 2)

cat("平均成長率:", round(mean(growth), 2), "% / 月\n")

### 練習問題 3

`co2` について次を求めてください。

1. 前年同月差 `diff(co2, lag = 12)` をプロットする（CO2 は毎年どのくらい増えている？）
2. その平均値を表示する（単位：ppm / 年）

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```r
yoy <- diff(co2, lag = 12)
plot(yoy, main = "CO2: year-over-year change", ylab = "ppm / year", col = "seagreen")
abline(h = mean(yoy), col = "red", lty = 2)
cat("平均:", round(mean(yoy), 2), "ppm / 年\n")
```

</details>

---
## 4. 移動平均 — なめらかな傾向を見る

**移動平均** は前後の値を平均して細かい変動をならし、大きな傾向を見やすくする方法です。
base R では `stats::filter()` を使います（dplyr を読み込んでいると `filter` は行の絞り込みに
なってしまうので、**`stats::filter()` とパッケージ名を付けて** 呼ぶのが安全です）。

```r
stats::filter(x, rep(1/12, 12), sides = 2)   # 中心化 12 か月移動平均
```

`rep(1/12, 12)` は「12 個の値に 1/12 ずつの重み」という意味で、`sides = 2` は前後対称に取ります。
端の 11 か月分は計算できないため `NA` になります。

In [ ]:
ma12 <- stats::filter(AirPassengers, rep(1 / 12, 12), sides = 2)

plot(AirPassengers, col = "gray60", main = "12-month moving average",
     ylab = "Passengers (1000s)")
lines(ma12, col = "red", lwd = 2)
legend("topleft", legend = c("Original", "12-month MA"),
       col = c("gray60", "red"), lwd = c(1, 2))

### 練習問題 4

2 章の練習で作った四半期売上 `q`（無ければ作り直してください）に
**4 四半期移動平均** をかけ、原系列（灰色）と移動平均（赤・太線）を重ねてプロットしてください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```r
q <- ts(c(120, 135, 160, 210, 128, 149, 175, 231),
        start = c(2024, 1), frequency = 4)
ma4 <- stats::filter(q, rep(1 / 4, 4), sides = 2)
plot(q, col = "gray60", main = "Quarterly sales with 4Q moving average",
     ylab = "Sales")
lines(ma4, col = "red", lwd = 2)
```

</details>

---
## 5. 分解 — トレンド・季節・残差に分ける

時系列は多くの場合、次の 3 つの成分の組み合わせとして理解できます。

- **トレンド**（長期的な傾向）
- **季節成分**（毎年繰り返すパターン）
- **残差**（残りの不規則な変動）

`decompose()` は移動平均ベースの古典的な分解です。成分の合わせ方は 2 通りあります。

| type | 式 | 向いている系列 |
|---|---|---|
| `"additive"` | データ = トレンド + 季節 + 残差 | 振れ幅が一定 |
| `"multiplicative"` | データ = トレンド × 季節 × 残差 | 振れ幅が水準に比例して拡大（AirPassengers 型） |

より柔軟な `stl()`（Loess による分解、季節の形が年々変わってもよい）もよく使われます。

In [ ]:
dec <- decompose(AirPassengers, type = "multiplicative")
plot(dec)   # 上から: 原系列 / トレンド / 季節 / 残差

In [ ]:
# 季節成分（月ごとの倍率）を取り出す
factors <- round(tapply(dec$seasonal, cycle(AirPassengers), mean), 3)
names(factors) <- month.abb
print(factors)   # 7 月・8 月は年平均の 1.2 倍以上、11 月は 0.8 倍程度

barplot(factors, main = "Seasonal factors (multiplicative)",
        ylab = "Factor", col = "steelblue")
abline(h = 1, col = "red", lty = 2)

In [ ]:
# stl()：Loess による分解（log をとって加法分解にするのが定石）
fit_stl <- stl(log(AirPassengers), s.window = "periodic")
plot(fit_stl, main = "STL decomposition of log(AirPassengers)")

### 練習問題 5

`co2` を `decompose()` で **加法分解**（振れ幅がほぼ一定なので additive）し、

1. 分解結果をプロットする
2. 月ごとの季節成分の平均を求め、**最も CO2 が高くなる月** を表示する
   （ヒント：`tapply(dec_co2$seasonal, cycle(co2), mean)` と `which.max()`、`month.abb`）

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```r
dec_co2 <- decompose(co2, type = "additive")
plot(dec_co2)

s <- tapply(dec_co2$seasonal, cycle(co2), mean)
names(s) <- month.abb
print(round(s, 2))
cat("最も高い月:", month.abb[which.max(s)], "\n")   # 北半球の春（5 月）がピーク
```

</details>

---
## 6. 自己相関 — 過去との相関を測る

**自己相関（ACF）** は「今期の値と k 期前の値の相関」をラグ k ごとに並べたものです。

- `acf(x)`：自己相関。ゆっくり減衰 → トレンドあり、周期的な山 → 季節性あり
- `pacf(x)`：偏自己相関。途中のラグの影響を除いた「直接の」相関で、AR モデルの次数選びに使う

グラフの青い点線は「相関ゼロとみなせる範囲」の目安です。
ホワイトノイズ（完全にランダムな系列）なら、ほとんどの棒がこの範囲に収まります。

In [ ]:
par(mfrow = c(1, 2))
acf(AirPassengers, main = "ACF: AirPassengers")           # ゆっくり減衰 + 12 か月周期の山
acf(diff(log(AirPassengers)), main = "ACF: growth rate")  # 差分をとると相関は大きく減る
par(mfrow = c(1, 1))

In [ ]:
# ホワイトノイズと AR(1) の比較
set.seed(10)
wn  <- ts(rnorm(200))                                   # ホワイトノイズ
ar1 <- arima.sim(model = list(ar = 0.8), n = 200)       # 1 期前と相関 0.8 の AR(1)

par(mfrow = c(1, 2))
acf(wn, main = "ACF: white noise")
acf(ar1, main = "ACF: AR(1), phi = 0.8")
par(mfrow = c(1, 1))

### 練習問題 6

`pacf()` を使って、上で作った `ar1` の **偏自己相関** をプロットしてください。
AR(1) では「ラグ 1 だけが大きく、ラグ 2 以降はほぼゼロ」になることを確認しましょう。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```r
pacf(ar1, main = "PACF: AR(1), phi = 0.8")
# ラグ 1 の棒だけが青線を大きく超える → AR の次数は 1 と判断できる
```

</details>

---
## 7. ARIMA モデル — 当てはめて予測する

**ARIMA(p, d, q)** は時系列予測の代表的なモデルです。

- **AR(p)**：過去 p 期の自分自身で説明する
- **I(d)**：d 回差分をとって定常にする
- **MA(q)**：過去 q 期の予測誤差で説明する

季節性のある月次データには **季節 ARIMA** を使います。AirPassengers には、対数変換のうえで
`ARIMA(0,1,1)(0,1,1)[12]`（通称 **airline model**）を当てはめるのが古典的な定石です。

```r
fit <- arima(log(x), order = c(0, 1, 1),
             seasonal = list(order = c(0, 1, 1), period = 12))
predict(fit, n.ahead = 24)   # $pred（予測値）と $se（標準誤差）
```

モデル間の比較には **AIC**（小さいほど良い）を使います。

In [ ]:
fit <- arima(log(AirPassengers), order = c(0, 1, 1),
             seasonal = list(order = c(0, 1, 1), period = 12))
print(fit)

# 別のモデルと AIC で比較（小さい方が良い）
fit2 <- arima(log(AirPassengers), order = c(1, 1, 0),
              seasonal = list(order = c(1, 1, 0), period = 12))
cat("airline model の AIC:", round(AIC(fit), 1), "\n")
cat("AR 型モデルの AIC   :", round(AIC(fit2), 1), "\n")

In [ ]:
# 24 か月先まで予測して、95% 区間と一緒に描く（log の世界から exp() で戻す）
pr <- predict(fit, n.ahead = 24)
point <- exp(pr$pred)                    # 予測値
upper <- exp(pr$pred + 1.96 * pr$se)     # 上側 95%
lower <- exp(pr$pred - 1.96 * pr$se)     # 下側 95%

ts.plot(AirPassengers, point, col = c("black", "red"), lwd = c(1, 2),
        main = "ARIMA forecast (24 months)", ylab = "Passengers (1000s)")
lines(upper, col = "red", lty = 2)
lines(lower, col = "red", lty = 2)
legend("topleft", legend = c("Observed", "Forecast", "95% interval"),
       col = c("black", "red", "red"), lty = c(1, 1, 2), lwd = c(1, 2, 1))

### 練習問題 7

内蔵データ `Nile`（ナイル川の年間流量、年次データ）に `arima(Nile, order = c(1, 1, 1))` を当てはめ、

1. モデルの中身と AIC を表示する
2. 10 年先まで予測し、観測値（黒）と予測値（赤・太線）、95% 区間（赤・点線）をプロットする
   （年次データなので対数変換や季節項は不要です）

In [ ]:
# 練習問題 7 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 7 の解答例を見る</strong></summary>

```r
fit_nile <- arima(Nile, order = c(1, 1, 1))
print(fit_nile)
cat("AIC:", round(AIC(fit_nile), 1), "\n")

pr <- predict(fit_nile, n.ahead = 10)
ts.plot(Nile, pr$pred, col = c("black", "red"), lwd = c(1, 2),
        main = "Nile: ARIMA(1,1,1) forecast", ylab = "Flow")
lines(pr$pred + 1.96 * pr$se, col = "red", lty = 2)
lines(pr$pred - 1.96 * pr$se, col = "red", lty = 2)
```

</details>

---
## 8. 指数平滑法 — HoltWinters

**指数平滑法** は「新しい観測ほど大きな重みで平均する」予測法で、実務の売上予測などで定番です。
`HoltWinters()` は水準・トレンド・季節の 3 成分を同時に更新します
（季節性が倍率で効くデータには `seasonal = "multiplicative"`）。

`predict()` に `prediction.interval = TRUE` を付けると予測区間も計算され、
`plot(モデル, 予測)` で観測値・あてはめ値・予測をまとめて描けます。

In [ ]:
hw <- HoltWinters(AirPassengers, seasonal = "multiplicative")
cat("alpha（水準）:", round(hw$alpha, 3),
    " beta（トレンド）:", round(hw$beta, 3),
    " gamma（季節）:", round(hw$gamma, 3), "\n")

pr_hw <- predict(hw, n.ahead = 24, prediction.interval = TRUE, level = 0.95)
print(head(pr_hw))   # fit（予測値）・upr・lwr の 3 列

plot(hw, pr_hw, main = "Holt-Winters forecast (24 months)",
     ylab = "Passengers (1000s)")

### 練習問題 8

`co2` に `HoltWinters()`（季節は既定の additive のまま）を当てはめ、
**36 か月先まで** 予測区間付きで予測し、`plot(モデル, 予測)` で描いてください。

In [ ]:
# 練習問題 8 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 8 の解答例を見る</strong></summary>

```r
hw_co2 <- HoltWinters(co2)
pr_co2 <- predict(hw_co2, n.ahead = 36, prediction.interval = TRUE)
plot(hw_co2, pr_co2, main = "CO2: Holt-Winters forecast (36 months)",
     ylab = "ppm")
```

</details>

---
## 9. ランダムウォーク — 「見せかけのトレンド」に注意

**ランダムウォーク** は「今期の値 = 前期の値 + ランダムな変化」という系列です。
純粋な偶然の積み重ねなのに、**トレンドがあるように見えてしまう** のが重要なポイントで、
株価などの分析で「上がり続けているから来期も上がる」と考える危うさを教えてくれます。

- ランダムウォークの ACF はゆっくりとしか減衰しない（過去を引きずる）
- しかし **差分をとればただのホワイトノイズ**（ACF はほぼゼロ）

「差分をとると定常になる」系列を **単位根を持つ** と呼び、ARIMA の d = 1 に対応します。

In [ ]:
set.seed(42)
rw <- ts(cumsum(rnorm(300)))   # ランダムウォーク：変化量 rnorm の累積和

par(mfrow = c(1, 2))
plot(rw, main = "Random walk", ylab = "Value", col = "steelblue")
plot(diff(rw), main = "First difference", ylab = "Change", col = "tomato")
par(mfrow = c(1, 1))

In [ ]:
par(mfrow = c(1, 2))
acf(rw, main = "ACF: random walk")          # ゆっくり減衰
acf(diff(rw), main = "ACF: difference")     # ほぼゼロ（ホワイトノイズ）
par(mfrow = c(1, 1))

### 練習問題 9

`set.seed(1)` で乱数を固定し、**300 期のランダムウォークを 5 本** 生成して
1 つのグラフに重ねてください（ヒント：`replicate(5, cumsum(rnorm(300)))` で 300×5 の行列を作り、
`matplot(..., type = "l")` で描く）。同じ仕組みから生まれた系列が、
上昇「トレンド」に見えたり下降に見えたり、バラバラの姿になることを確認しましょう。

In [ ]:
# 練習問題 9 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 9 の解答例を見る</strong></summary>

```r
set.seed(1)
paths <- replicate(5, cumsum(rnorm(300)))
matplot(paths, type = "l", lty = 1, lwd = 1.5,
        main = "Five random walks", xlab = "Time", ylab = "Value")
abline(h = 0, col = "gray40", lty = 2)
# どれも「意味のある傾向」に見えるが、実体はただの偶然の積み重ね
```

</details>

---
## まとめ

| 章 | 学んだこと |
|---|---|
| ts | `ts(start=, frequency=)`、`window()`、`start()` / `end()` / `cycle()` |
| 可視化 | `plot()`、`ts.plot()`、対数変換で振れ幅を一定に |
| 差分 | `diff()`（前期差・前年差）、`diff(log(x))` ≒ 成長率、`lag()` |
| 平滑 | `stats::filter()` による移動平均 |
| 分解 | `decompose(type=)`、`stl(s.window=)`、季節成分の読み方 |
| 自己相関 | `acf()` / `pacf()`、ホワイトノイズとの見分け方 |
| モデル | `arima()`、季節 ARIMA、`AIC()`、`predict(n.ahead=)` |
| 予測 | `HoltWinters()` + `predict(prediction.interval=)` |
| 落とし穴 | ランダムウォークの見せかけのトレンド、単位根と差分 |

## 次のステップ
- `r/r_dplyr_tidyr_beginner_tutorial.ipynb` — データ前処理入門（集計してから ts にする、が実務の流れ）
- `jupyterlite/jupyterlite_xeus_r_stats_practice.ipynb` — 統計テスト演習
- 本格的な予測パッケージ（forecast / fable など）はこの環境にはありませんが、
  ここで学んだ ts・分解・ARIMA の考え方はそのまま通用します

---
## 総合演習 — 月次売上の分解と 12 か月先予測

架空のオンラインストア 8 年分（96 か月）の月次売上データを分析します。

**課題**：次のセルで作られる `sales_ts` について、

1. 全期間をプロットして、トレンドと季節性を目視で確認する
2. `decompose()`（乗法型）で分解してプロットし、**最も売上が高くなる月** を求める
3. `HoltWinters()`（乗法季節）で 12 か月先まで予測区間付きで予測し、プロットする
4. 12 か月先予測の **合計**（来年 1 年間の売上見込み）を計算する

まず自分で書いてから、解答例と見比べてください。

In [ ]:
# 月次売上データの生成（このセルを実行してから取り組んでください）
set.seed(123)
n_months <- 96
trend  <- seq(100, 180, length.out = n_months)                       # 緩やかな成長
season <- rep(c(0.90, 0.92, 1.00, 1.02, 1.05, 1.08,
                1.20, 1.15, 1.02, 0.98, 0.95, 1.25), times = 8)      # 12 月商戦・夏に山
sales_ts <- ts(round(trend * season * exp(rnorm(n_months, 0, 0.03))),
               start = c(2018, 1), frequency = 12)

print(window(sales_ts, end = c(2018, 12)))   # 最初の 1 年分を確認

<details>
<summary><strong>総合演習の解答例を見る</strong></summary>

```r
# 1. 全期間のプロット
plot(sales_ts, main = "Monthly sales (2018-2025)", ylab = "Sales", col = "steelblue")

# 2. 乗法分解と「売上が最も高い月」
dec_s <- decompose(sales_ts, type = "multiplicative")
plot(dec_s)
f <- tapply(dec_s$seasonal, cycle(sales_ts), mean)
names(f) <- month.abb
print(round(f, 3))
cat("最も売上が高い月:", month.abb[which.max(f)], "\n")   # 12 月

# 3. Holt-Winters による 12 か月先予測
hw_s <- HoltWinters(sales_ts, seasonal = "multiplicative")
pr_s <- predict(hw_s, n.ahead = 12, prediction.interval = TRUE)
plot(hw_s, pr_s, main = "Sales forecast (12 months)", ylab = "Sales")

# 4. 来年 1 年間の売上見込み
cat("12 か月先予測の合計:", round(sum(pr_s[, "fit"])), "\n")
```

</details>

お疲れさまでした！ 「プロット → 分解 → モデル → 予測」という流れは、
どの時系列データでも使える基本の型です。まずは身近な月次データ（売上・アクセス数・電気代など）を
`ts()` にして、同じ手順をなぞってみてください。